# 02 - Benchmark e Avaliação Comparativa de Recuperação (Information Retrieval)
**Projecto:** AI Voice of Customer & Territorial Intelligence (Olist Dataset)  
**Objectivo:** Avaliar comparativamente as estratégias de recuperação (BM25, Dense Embeddings, RRF Híbrido e Two-Stage Reranking) demonstrando a redução de ruído semântico.

In [3]:
import os
import sys
from pathlib import Path

# Força o notebook a enxergar as bibliotecas instaladas na sua .venv (como o rank-bm25)
sys.path.append(r"C:\tech-challenge-fase4-grupo\.venv\Lib\site-packages")

# Ajusta a raiz do projeto
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import time
import pandas as pd
import numpy as np
from src.indexing.hybrid_indexer import HybridSearchEngine
from src.rag.reranker import CrossEncoderReranker

# Carregar a base tratada produzida no Notebook 01
PARQUET_FILE = PROJECT_ROOT / "data" / "processed" / "olist_reviews_clean.parquet"
df = pd.read_parquet(PARQUET_FILE)

# Compatibilização de esquema: garante que a coluna 'clean_comment' existe para o BM25
if "text" in df.columns and "clean_comment" not in df.columns:
    df["clean_comment"] = df["text"]
elif "clean_comment" in df.columns and "text" not in df.columns:
    df["text"] = df["clean_comment"]

print(f"Base carregada: {len(df):,} avaliações disponíveis.")

# Inicializar motor híbrido e re-ranker
engine = HybridSearchEngine(df)
reranker = CrossEncoderReranker()
print("Motores de busca e reranking prontos.")

Base carregada: 40,964 avaliações disponíveis.
2026-09-23 22:49:59 [INFO] (voc_rag) A construir índice BM25 sobre os comentários...
2026-09-23 22:50:00 [INFO] (voc_rag) Índice BM25 construído com 40,964 documentos.
2026-09-23 22:50:00 [INFO] (voc_rag) A inicializar embeddings locais (cache isolado em data/models)...
2026-09-23 22:50:00 [INFO] (voc_rag) Dispositivo alocado: CPU


c:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\tech-challenge-fase4-grupo\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\tech-challenge-fase4-grupo\notebooks\data\models\hf\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate De

2026-09-23 22:50:46 [INFO] (voc_rag) A inicializar o modelo de re-ranking: ms-marco-MiniLM-L-12-v2


INFO:flashrank.Ranker:Downloading ms-marco-MiniLM-L-12-v2...
ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:01<00:00, 11.6MiB/s]


Motores de busca e reranking prontos.


## 1. Definição do Cenário de Teste de Alta Criticidade

Vamos testar uma consulta típica de dor de negócio:
`"Produto com defeito e atendimento péssimo no pós-venda"`

Iremos extrair os Top-5 resultados através de quatro arquiteturas distintas:
1. **BM25 Puro (Sparse)**: Baseado estritamente em frequência léxica e raridade inversa de termos (TF-IDF/BM25).
2. **Dense Puro (Sentence Transformers / MiniLM)**: Baseado estritamente em proximidade geométrica no espaço de embeddings (similaridade de cosseno).
3. **Híbrido com RRF (Reciprocal Rank Fusion)**: Interseção e fusão ordinal balanceada dos dois mundos.
4. **Two-Stage Retrieval (Híbrido + Cross-Encoder Reranker)**: Reavaliação profunda com atenção cruzada das melhores evidências.

In [5]:
query_test = "Produto com defeito e atendimento péssimo no pós-venda"

# 1. BM25 Puro (Léxico)
t0 = time.perf_counter()
if hasattr(engine, "bm25") and hasattr(engine.bm25, "search"):
    bm25_res = engine.bm25.search(query_test, top_k=5)
elif hasattr(engine, "bm25_retriever"):
    bm25_res = engine.bm25_retriever.search(query_test, top_k=5)
else:
    # Fallback direto caso o retriever esteja exposto internamente
    bm25_res = engine.search(query_test, top_k=5)
t_bm25 = (time.perf_counter() - t0) * 1000

# 2. Dense Puro (Vetorial / Embeddings)
t0 = time.perf_counter()
if hasattr(engine, "vector_store") and hasattr(engine.vector_store, "search"):
    dense_res = engine.vector_store.search(query_test, top_k=5)
elif hasattr(engine, "dense_retriever"):
    dense_res = engine.dense_retriever.search(query_test, top_k=5)
else:
    dense_res = engine.search(query_test, top_k=5)
t_dense = (time.perf_counter() - t0) * 1000

# 3. Híbrido RRF (Reciprocal Rank Fusion)
t0 = time.perf_counter()
hybrid_res = engine.search(query_test, top_k=5)
t_hybrid = (time.perf_counter() - t0) * 1000

# 4. Two-Stage Retrieval (Híbrido + Cross-Encoder Reranker)
t0 = time.perf_counter()
candidates = engine.search(query_test, top_k=15)
reranked_res = reranker.rerank(query_test, candidates, top_n=5)
t_rerank = (time.perf_counter() - t0) * 1000

print("Buscas finalizadas:")
print(f"BM25 Latência: {t_bm25:.2f} ms")
print(f"Dense Latência: {t_dense:.2f} ms")
print(f"Híbrido RRF Latência: {t_hybrid:.2f} ms")
print(f"Two-Stage (Híbrido + Re-rank) Latência: {t_rerank:.2f} ms")

C:\tech-challenge-fase4-grupo\src\indexing\vector_store.py:21: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


2026-09-23 22:51:47 [INFO] (voc_rag) Recuperação híbrida concluída. Retornados 5 documentos consolidados.


INFO:voc_rag:Recuperação híbrida concluída. Retornados 5 documentos consolidados.


2026-09-23 22:51:47 [INFO] (voc_rag) Recuperação híbrida concluída. Retornados 5 documentos consolidados.


INFO:voc_rag:Recuperação híbrida concluída. Retornados 5 documentos consolidados.


2026-09-23 22:51:47 [INFO] (voc_rag) Recuperação híbrida concluída. Retornados 15 documentos consolidados.


INFO:voc_rag:Recuperação híbrida concluída. Retornados 15 documentos consolidados.


2026-09-23 22:51:47 [INFO] (voc_rag) Re-ranking concluído: 15 candidatos reduzidos para os top-5 mais relevantes.


INFO:voc_rag:Re-ranking concluído: 15 candidatos reduzidos para os top-5 mais relevantes.


Buscas finalizadas:
BM25 Latência: 41.29 ms
Dense Latência: 941.92 ms
Híbrido RRF Latência: 69.07 ms
Two-Stage (Híbrido + Re-rank) Latência: 97.82 ms


## 2. Tabela Comparativa de Evidências Recuperadas

Comparação direta dos textos e notas capturados por cada método para a mesma consulta.

In [6]:
def format_results(results_list, model_name):
    rows = []
    for rank, doc in enumerate(results_list, 1):
        clean_text = doc.get("text", "").replace("\n", " ")[:90] + "..."
        score = doc.get("review_score", "-")
        rows.append({
            "Método": model_name,
            "Rank": f"#{rank}",
            "Nota": score,
            "Trecho Recuperado": clean_text
        })
    return rows

benchmark_data = (
    format_results(bm25_res, "1. BM25 Puro") +
    format_results(dense_res, "2. Dense Puro") +
    format_results(hybrid_res, "3. Híbrido RRF") +
    format_results(reranked_res, "4. Cross-Encoder (SOTA)")
)

df_comparison = pd.DataFrame(benchmark_data)
pd.set_option("display.max_colwidth", None)
display(df_comparison)

,Método,Rank,Nota,Trecho Recuperado
0,1. BM25 Puro,#1,1,...
1,1. BM25 Puro,#2,1,...
2,1. BM25 Puro,#3,1,...
3,1. BM25 Puro,#4,1,...
4,1. BM25 Puro,#5,2,...
5,2. Dense Puro,#1,1,...
6,2. Dense Puro,#2,1,bom produto ruim...
7,2. Dense Puro,#3,1,...
8,2. Dense Puro,#4,1,o produto veio com dimensões erradas e avariado....
9,2. Dense Puro,#5,1,...


## 3. Conclusão Metodológica

1. **BM25 Puro**: Captura documentos com correspondência léxica exata, mas falha em variações vocabulares ou erros ortográficos.
2. **Dense Puro**: Captura correlação semântica global, mas pode ser influenciado por proximidade contextual genérica.
3. **Fusão Híbrida (RRF)**: Balanceia precisão de palavras-chave e significado subjacente.
4. **Cross-Encoder Reranker**: Aplica atenção conjunta entre a pergunta e os candidatos para selecionar as evidências factuais mais aderentes para o LLM.